# Role 3 — DistilBERT Classification Notebook
Team 7 — IE7374: Generative AI  
Role 3 — Umang Khamar (NLP Engineer)

This notebook begins where Role 2 (EDA) ends.  
We load the processed dataset created by `preprocessing.py` and enriched by Role 2.

Before anything else, we clone the project repository so that all required files
(`df_clean.csv`, `df_focused.csv`, configs, src scripts, etc.) are available in the Colab runtime.

In [7]:
# ============================================================
# Clone the project repository into Colab
# ============================================================

!git clone https://github.com/AHonGa/IE7374-Group-Project-07-.git

# Move into the repo directory
%cd IE7374-Group-Project-07-

print("Repository cloned and ready.")

Cloning into 'IE7374-Group-Project-07-'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 43 (delta 0), reused 0 (delta 0), pack-reused 41 (from 1)
Receiving objects: 100% (43/43), 42.20 MiB | 19.15 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Updating files: 100% (21/21), done.
/content/IE7374-Group-Project-07-/IE7374-Group-Project-07-
Repository cloned and ready.


## Verify Repository Folder Structure

Before loading any data, we verify that the repository was cloned correctly
and that all required folders and files exist.

This step ensures:
- `data/processed/df_focused.csv` is available for Role 3
- `configs/model_config.yaml` exists
- `notebooks/` contains Role 1 and Role 2 notebooks
- `experiments/` and `models/` are ready for Role 3 outputs

In [8]:
# ============================================================
# Verify repository folder structure
# ============================================================

import os

for root, dirs, files in os.walk(".", topdown=True):
    level = root.count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

print("\nFolder structure verified.")

./
  requirements.txt
  .gitattributes
  README.md
  .gitignore
  utils/
    helpers.py
    __init__.py
  notebooks/
    eda.ipynb
    project_team_7.ipynb
  configs/
    model_config.yaml
  outputs/
    .gitkeep
  experiments/
    .gitkeep
  data/
    processed/
      drugsComTest_raw.csv
      drugsComTrain_raw.csv
      df_focused.csv
      df_clean.csv
  .git/
    packed-refs
    HEAD
    index
    config
    description
    objects/
      info/
      pack/
        pack-764a4c61c1712e37229653e320f43ede7d43b368.pack
        pack-764a4c61c1712e37229653e320f43ede7d43b368.idx
    refs/
      heads/
        main
      tags/
      remotes/
        origin/
          HEAD
    hooks/
      pre-push
      pre-commit.sample
      pre-applypatch.sample
      update.sample
      post-update.sample
      commit-msg.sample
      pre-receive.sample
      prepare-commit-msg.sample
      pre-merge-commit.sample
      pre-rebase.sample
      fsmonitor-watchman.sample
      pre-push.sample
      push-

## Load Focused Dataset

Role 2 (EDA) produced two processed datasets:

- `df_clean.csv` — cleaned full dataset  
- `df_focused.csv` — subset containing only Depression + Anxiety reviews  
  enriched with:
  - `review_clean` — cleaned text for transformer models  
  - `review_processed` — lemmatized text  
  - `vader_score` — VADER compound sentiment  
  - `sentiment_label` — positive / neutral / negative  
  - `topic_id` — LDA topic assignment  

This dataset is the starting point for Role 3 (DistilBERT classification).

In [9]:
# ============================================================
# Load the focused dataset created by Role 2 (EDA)
# ============================================================

import pandas as pd

df = pd.read_csv("data/processed/df_focused.csv")

print("Focused dataset loaded successfully.")
print("Shape:", df.shape)

# Display first few rows to confirm structure
df.head()

Focused dataset loaded successfully.
Shape: (11814, 11)


,uniqueID,drugName,condition,review,rating,date,usefulCount,split,review_clean,review_word_count,review_processed
0,75612,L-Methylfolate,Depression,"""I have taken anti-depressants for years, with...",10,2017-03-09,54,train,"I have taken anti-depressants for years, with ...",80,taken year improvement mostly moderate severe ...
1,96233,Sertraline,Depression,"""1 week on Zoloft for anxiety and mood swings....",8,2011-05-07,3,train,1 week on Zoloft for anxiety and mood swings. ...,51,week zoloft anxiety mood swing take morning br...
2,121333,Venlafaxine,Depression,"""my gp started me on Venlafaxine yesterday to ...",4,2016-04-27,3,train,my gp started me on Venlafaxine yesterday to h...,136,gp started venlafaxine yesterday help depressi...
3,131704,Effexor Xr,Anxiety,"""Was on this med for 5 years. Worked fine but ...",6,2016-12-27,23,train,Was on this med for 5 years. Worked fine but n...,62,med year worked fine not great stopped panic a...
4,131909,Effexor Xr,Depression,"""This medicine saved my life. I was at my wits...",10,2013-06-20,166,train,This medicine saved my life. I was at my wits ...,106,medicine saved life wit end ready give doctor ...


## Rebuild Missing Columns (VADER, Sentiment Label, Topic ID)

The `df_focused.csv` in the repository does not contain the enriched columns
produced in Role 2 (EDA). To ensure Role 3 has the correct dataset, we regenerate:

- `vader_score` — VADER compound sentiment  
- `sentiment_label` — positive / neutral / negative  
- `topic_id` — dominant LDA topic (0–7)

This cell reconstructs these columns using the same logic as the EDA notebook,
but without generating plots.

In [11]:
# ============================================================
# Rebuild missing Role 2 columns: vader_score, sentiment_label, topic_id
# ============================================================

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

nltk.download("vader_lexicon", quiet=True)

# -----------------------------
# 1. VADER sentiment scoring
# -----------------------------
sia = SentimentIntensityAnalyzer()

print("Computing VADER scores...")
df["vader_score"] = df["review_clean"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

def label_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

df["sentiment_label"] = df["vader_score"].apply(label_sentiment)

print("Added vader_score and sentiment_label.")


# -----------------------------
# 2. LDA topic modeling
# -----------------------------
print("Running LDA topic modeling...")

vectorizer = CountVectorizer(
    max_features=3000,
    min_df=5,
    max_df=0.95,
    ngram_range=(1, 2)
)

corpus = df["review_processed"].dropna()
dtm = vectorizer.fit_transform(corpus)

lda = LatentDirichletAllocation(
    n_components=8,
    random_state=42,
    max_iter=15,
    learning_method="online"
)
lda.fit(dtm)

topic_dist = lda.transform(dtm)
dominant_topic = topic_dist.argmax(axis=1)

# Align indices
corpus_index = df["review_processed"].dropna().index
df.loc[corpus_index, "topic_id"] = dominant_topic

print("Added topic_id column.")


# -----------------------------
# Save updated df_focused.csv
# -----------------------------
df.to_csv("data/processed/df_focused.csv", index=False)
print("Updated df_focused.csv saved successfully.")

Computing VADER scores...
Added vader_score and sentiment_label.
Running LDA topic modeling...
Added topic_id column.
Updated df_focused.csv saved successfully.


## Encode Sentiment Labels for DistilBERT

DistilBERT requires numeric labels for classification.  
We convert the categorical `sentiment_label` column (positive / neutral / negative)
into integer IDs:

- negative → 0  
- neutral → 1  
- positive → 2  

This mapping will be used throughout the training and evaluation pipeline.


In [12]:
# ============================================================
# Encode sentiment labels into numeric IDs for DistilBERT
# ============================================================

label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

df["label_id"] = df["sentiment_label"].map(label_map)

print("Label encoding complete.")
df[["sentiment_label", "label_id"]].head()

Label encoding complete.


,sentiment_label,label_id
0,neutral,1
1,positive,2
2,negative,0
3,negative,0
4,positive,2


## Sample 500 Reviews for Preliminary Experiment

Role 3 requires only a *short-run* DistilBERT experiment to validate the
end‑to‑end pipeline (tokenization → dataloader → model → evaluation).

To keep training fast and lightweight, we randomly sample **500 reviews**
from the enriched focused dataset.

This subset is used for:
- sanity‑check training
- quick evaluation (accuracy + F1)
- saving a preliminary model
- generating the experiment report

Full training can be done later if needed.

In [13]:
# ============================================================
# Sample 500 reviews for the preliminary DistilBERT experiment
# ============================================================

df_sample = df.sample(n=500, random_state=42).reset_index(drop=True)

print("Sampled dataset shape:", df_sample.shape)
df_sample.head()

Sampled dataset shape: (500, 15)


,uniqueID,drugName,condition,review,rating,date,usefulCount,split,review_clean,review_word_count,review_processed,vader_score,sentiment_label,topic_id,label_id
0,173184,Clonazepam,Anxiety,"""I take about 2-4 mg a day of Klonopin. It mak...",6,2011-04-27,2,test,I take about 2-4 mg a day of Klonopin. It make...,71,take mg day klonopin make tired sometimes work...,0.8658,positive,6.0,2
1,121093,Venlafaxine,Depression,"""Have been taking this for over 5 yrs and have...",8,2017-01-20,9,train,Have been taking this for over 5 yrs and have ...,62,taking yr found pretty good overall no side ef...,0.6900,positive,2.0,2
2,49726,Gabapentin,Anxiety,"""I&#039;m so sad reading all these great revie...",2,2016-09-01,44,train,I'm so sad reading all these great reviews . I...,143,sad reading great review odd one first time to...,-0.9034,negative,6.0,0
3,50537,Gabapentin,Anxiety,"""I suffered with severe migraines several time...",10,2011-11-20,65,train,I suffered with severe migraines several times...,99,suffered severe migraine several time year mig...,-0.6286,negative,1.0,0
4,45580,Fluoxetine,Depression,"""Prozac has allowed me to enjoy life and handl...",9,2012-08-03,9,train,Prozac has allowed me to enjoy life and handle...,127,prozac allowed enjoy life handle stressful sit...,0.8682,positive,7.0,2


## Load DistilBERT Tokenizer

The tokenizer converts raw text (`review_clean`) into numerical token IDs
and attention masks that DistilBERT can process.

We use:
- `distilbert-base-uncased` — lowercase English model
- `DistilBertTokenizerFast` — optimized Rust-backed tokenizer

This tokenizer will be used inside our PyTorch Dataset class.

In [14]:
# ============================================================
# Load DistilBERT tokenizer
# ============================================================

from transformers import DistilBertTokenizerFast

# Load pretrained tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

print("Tokenizer loaded successfully.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully.


## Build PyTorch Dataset and DataLoader

DistilBERT expects inputs in a specific tensor format:
- `input_ids` — token IDs
- `attention_mask` — mask for padded tokens
- `labels` — numeric sentiment label

To supply these to the model, we create a custom PyTorch `Dataset` class that:
1. Reads each review (`review_clean`)
2. Tokenizes it using the DistilBERT tokenizer
3. Packages the tensors into a dictionary
4. Adds the corresponding `label_id`

We then wrap this dataset in a `DataLoader` to enable batching and shuffling.

In [15]:
# ============================================================
# Build PyTorch Dataset + DataLoader for DistilBERT
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader

class ReviewsDataset(Dataset):
    """
    Custom dataset for DistilBERT sentiment classification.
    Each item returns:
    - tokenized review (input_ids, attention_mask)
    - label_id (0, 1, 2)
    """
    def __init__(self, df):
        self.texts = df["review_clean"].tolist()
        self.labels = df["label_id"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Tokenize the review text
        encoded = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        # Convert token outputs from shape (1, seq_len) → (seq_len)
        item = {key: val.squeeze(0) for key, val in encoded.items()}

        # Add label tensor
        item["labels"] = torch.tensor(self.labels[idx])

        return item

# Create dataset and dataloader
dataset = ReviewsDataset(df_sample)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

print("Dataset and DataLoader created successfully.")

Dataset and DataLoader created successfully.


## Load DistilBERT Model for Sentiment Classification

We now load the pretrained DistilBERT model:

- `distilbert-base-uncased`  
- configured for **3 sentiment classes**:
  - negative (0)
  - neutral (1)
  - positive (2)

This model will be fine‑tuned on our 500‑sample subset to validate the full
training pipeline.

The model outputs:
- logits → raw class scores
- loss → cross‑entropy loss (when labels are provided)

This cell prepares the model for training.

In [16]:
# ============================================================
# Load DistilBERT model for sequence classification
# ============================================================

from transformers import DistilBertForSequenceClassification

# Load pretrained DistilBERT with 3 output labels
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

print("DistilBERT model loaded successfully.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded successfully.


## Sanity Check Training (Single Batch)

Before running any full training loop, we perform a **sanity check**:

- Take **one batch** from the DataLoader  
- Run it through DistilBERT  
- Compute the loss  
- Backpropagate  
- Update model weights  

This verifies that:
- tokenization works  
- tensors have correct shapes  
- the model receives inputs correctly  
- loss computation is functioning  
- optimizer updates parameters  

If this cell runs without errors, the pipeline is ready for short training.

In [17]:
# ============================================================
# Sanity check: train on a single batch
# ============================================================

import torch

# AdamW is standard for transformer fine-tuning
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()

# Take one batch from the DataLoader
for batch in loader:
    optimizer.zero_grad()

    # Forward pass → compute loss
    outputs = model(**batch)
    loss = outputs.loss

    # Backpropagation
    loss.backward()

    # Update model weights
    optimizer.step()

    print(f"Sanity check loss: {loss.item():.4f}")
    break  # Only one batch

Sanity check loss: 1.0736


## Evaluate DistilBERT (Accuracy + F1)

After the sanity‑check batch, we run a lightweight evaluation on the 500‑sample
subset. This is not a full training run — just a quick check to ensure:

- the model produces logits correctly  
- predictions map to valid label IDs  
- accuracy and F1 can be computed end‑to‑end  

This confirms the pipeline is ready for short fine‑tuning.

In [18]:
# ============================================================
# Lightweight evaluation: accuracy + macro F1
# ============================================================

from sklearn.metrics import accuracy_score, f1_score

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in loader:
        outputs = model(**batch)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1).cpu().numpy()
        labels = batch["labels"].cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels)

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {f1:.4f}")

Accuracy: 0.4740
Macro F1: 0.2367


## Short Training Loop (2 Epochs)

We now run a lightweight fine‑tuning loop over the 500‑sample dataset.

This is intentionally small‑scale:
- 2 epochs
- batch size = 16
- learning rate = 5e‑5

Purpose:
- validate full training pipeline
- produce a usable preliminary DistilBERT model
- generate metrics for the experiment report

This is **not** meant to be a full production training run.

In [19]:
# ============================================================
# Short training loop: 2 epochs
# ============================================================

import torch
from tqdm.auto import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
model.train()

EPOCHS = 2

for epoch in range(EPOCHS):
    loop = tqdm(loader, leave=True)
    total_loss = 0

    for batch in loop:
        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}/{EPOCHS}")
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

  0%|          | 0/32 [00:00<?, ?it/s]

Epoch 1 average loss: 0.8638


  0%|          | 0/32 [00:00<?, ?it/s]

Epoch 2 average loss: 0.7283


## Final Evaluation After Training (Accuracy + F1)

After completing the short 2‑epoch training loop, we evaluate the model again
on the same 500‑sample subset.

This gives us:
- post‑training accuracy  
- post‑training macro‑F1  
- a sanity check that the model actually learned something  

These metrics will be included in the Role 3 experiment report.

In [20]:
# ============================================================
# Final evaluation after training
# ============================================================

from sklearn.metrics import accuracy_score, f1_score

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in loader:
        outputs = model(**batch)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1).cpu().numpy()
        labels = batch["labels"].cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels)

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Final Accuracy: {acc:.4f}")
print(f"Final Macro F1: {f1:.4f}")

Final Accuracy: 0.8120
Final Macro F1: 0.5484


## Save Preliminary DistilBERT Model + Tokenizer

After training, we save:
- the fine‑tuned DistilBERT model
- the tokenizer
- the training metadata (accuracy, F1)

These files will be stored under: models/distilbert_prelim/

This allows Role 4 (Summarization Engineer) to load the model for sentiment‑aware
summaries and reference training metrics in the final report.

In [21]:
# ============================================================
# Save preliminary DistilBERT model + tokenizer
# ============================================================

import os

save_dir = "models/distilbert_prelim"
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model and tokenizer saved to: {save_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to: models/distilbert_prelim


## Save Training Metrics (JSON)

To support reproducibility and experiment tracking, we save the key metrics from
the preliminary DistilBERT run:

- final accuracy  
- final macro‑F1  
- average loss per epoch  

These are stored in: models/distilbert_prelim/metrics.json

Role 4 (Summarization Engineer) will reference these metrics when designing
sentiment‑aware summarization prompts.

In [22]:
# ============================================================
# Save training metrics to JSON
# ============================================================

import json

metrics = {
    "final_accuracy": float(acc),
    "final_macro_f1": float(f1),
    "epoch_losses": {
        "epoch_1": 0.8638,
        "epoch_2": 0.7283
    }
}

metrics_path = "models/distilbert_prelim/metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print(f"Saved training metrics to: {metrics_path}")

Saved training metrics to: models/distilbert_prelim/metrics.json


## Experiment Summary (Preliminary DistilBERT Fine‑Tuning)

This section summarizes the short‑run DistilBERT experiment performed on a
500‑review subset of the enriched `df_focused` dataset.

### **Objective**
Validate the full end‑to‑end pipeline:
- preprocessing  
- tokenization  
- batching  
- model loading  
- training  
- evaluation  
- saving artifacts  

### **Dataset**
- 500 randomly sampled reviews  
- Labels: negative (0), neutral (1), positive (2)  
- Enriched with:
  - `vader_score`
  - `sentiment_label`
  - `topic_id`

### **Model**
- `distilbert-base-uncased`
- Fine‑tuned for 3‑class sentiment classification

### **Training Setup**
- 2 epochs  
- batch size = 16  
- learning rate = 5e‑5  
- optimizer = AdamW  

### **Results**
- **Final Accuracy:** 0.8120  
- **Final Macro F1:** 0.5484  
- **Epoch Losses:**
  - Epoch 1: 0.8638  
  - Epoch 2: 0.7283  

### **Artifacts Saved**
- Model: `models/distilbert_prelim/`  
- Tokenizer: `models/distilbert_prelim/`  
- Metrics: `models/distilbert_prelim/metrics.json`

### **Conclusion**
The preliminary experiment confirms that the DistilBERT pipeline is functioning
correctly. The model shows meaningful learning even with a small dataset,
producing solid accuracy and moderate macro‑F1. This setup is ready for:

- full‑scale fine‑tuning  
- integration into sentiment‑aware summarization prompts  
- downstream evaluation in Role 4  